# RadHarmony: Adding a Dataset the Production Way

Notebook 2 added a dataset **inline**: the harmonizer and dataset classes were
defined in a cell and registered on the fly. That is perfect for a quick
experiment, but the classes vanish when the kernel restarts.

This notebook does the same job the way you would to **contribute a dataset
back to RadHarmony**: it writes real files into the package, adds a JSON config,
wires up the exports, and runs the integrity check. When you are done, the
dataset loads by name in any fresh session, shows up in the Gradio app, and is
ready to open as a pull request.

We reuse the same toy **Montgomery County chest X-ray set** so you can compare
the two approaches side by side. The steps below mirror
[docs/add_dataset_skill.md](../docs/add_dataset_skill.md).


## The workflow

Every RadHarmony dataset is four small pieces plus a config and a check:

1. **JSON config** (`configs/datasets/<key>.json`): records every decision
   (paths, columns, label handling) in one place.
2. **Harmonizer** (`radharmony/harmonizer/<key>/`): reads the raw files and
   returns RadHarmony's canonical table.
3. **Dataset** (`radharmony/dataset/<key>/`): points at the harmonizer,
   registers a name, and inherits transforms / caching / splitting.
4. **Exports** (the two package `__init__.py` files, plus `app.py` for the
   Gradio browser).
5. **Integrity check**: harmonize, load a few samples through a `DataLoader`,
   and validate shapes / dtype / range / labels.

Each piece lives in its own folder so future variants (a train/test split, a
`Plus` release, an alternate reader) drop in without a restructure.


In [ ]:
import os
from pathlib import Path

import pandas as pd
import radharmony

# Locate the installed package so we write into the real source tree.
PKG_DIR = Path(radharmony.__file__).resolve().parent  # .../radharmony
REPO_ROOT = PKG_DIR.parent  # repository root
CONFIG_DIR = REPO_ROOT / "configs" / "datasets"

REGISTRY_KEY = "my_montgomery"
HARMONIZER_DIR = PKG_DIR / "harmonizer" / REGISTRY_KEY
DATASET_DIR = PKG_DIR / "dataset" / REGISTRY_KEY

# Toy data: the metadata CSV ships with the tutorial (relative to this folder);
# base_image_dir is the last common folder holding every PNG (same as notebook 1).
INDEX_CSV = "data/montgomery_index.csv"
MONTGOMERY_DIR = "/mnt/NAS4/datasets/external/Montgomery-CXR/MontgomerySet/CXR_png/"

for d in (CONFIG_DIR, HARMONIZER_DIR, DATASET_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("package tree:", PKG_DIR)
print("will create:")
for d in (CONFIG_DIR / f"{REGISTRY_KEY}.json", HARMONIZER_DIR, DATASET_DIR):
    print("  ", d)

package tree: /home/fli40/RadHarmony/radharmony
will create:
   /home/fli40/RadHarmony/configs/datasets/my_montgomery.json
   /home/fli40/RadHarmony/radharmony/harmonizer/my_montgomery
   /home/fli40/RadHarmony/radharmony/dataset/my_montgomery


## Step 0: the JSON config

Before writing code, capture the dataset in a config file. It is the single
place that records the paths, the column mapping, and how labels are handled,
and it doubles as documentation for the next person. Fill in everything you can
infer from the raw files; leave anything you truly cannot as `null`.


In [ ]:
config_json = r"""{
  "_comment": "Toy Montgomery County TB CXR set used in the RadHarmony datathon. A folder of PNG chest X-rays plus a small metadata CSV (data/montgomery_index.csv) with one row per image and columns pid / fname / tb. CSV-driven: the raw table already lists every image, so the harmonizer just maps columns. Full dataset source: https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Montgomery-County-CXR-Set/",
  "dataset_name": "My Montgomery (toy)",
  "registry_key": "my_montgomery",
  "modality": "2D",
  "image_format": "PNG",
  "base_image_dir": "<path_to_Montgomery>/MontgomerySet/CXR_png/",
  "primary_csv": {
    "_comment": "The metadata CSV that lists every image row.",
    "path": "data/montgomery_index.csv",
    "filename_variants": [
      "montgomery_index.csv"
    ]
  },
  "patient_id": {
    "_comment": "Direct column: pid. One study per patient in this set.",
    "column": "pid",
    "derivation": null
  },
  "study_id": {
    "_comment": "Same as patient_id; one study per patient.",
    "column": "pid",
    "derivation": "Mirror patient_id"
  },
  "image_path": {
    "_comment": "The fname column is already a bare filename relative to base_image_dir.",
    "column": "fname",
    "derivation": null,
    "prefix_to_strip": null
  },
  "labels": {
    "_comment": "Single binary TB label carried straight from the primary CSV.",
    "columns": [
      "tb"
    ],
    "separate_csv": {
      "_comment": "Label lives in the primary CSV; no separate file.",
      "path": null,
      "filename_variants": [],
      "join_columns": []
    },
    "uncertain_handling": {
      "_comment": "Binary 0/1 label with no uncertainty.",
      "nan_action": "keep",
      "negative_one_action": "keep",
      "drop_uncertain_rows": false
    }
  },
  "view_position": {
    "_comment": "All images are frontal CXRs; no view-position column.",
    "column": null,
    "source": null,
    "separate_csv": {
      "path": null,
      "filename_variants": [],
      "join_columns": []
    }
  },
  "masks": {
    "_comment": "Not exposed in this toy integration.",
    "supported": false,
    "column": null,
    "format": null
  },
  "bboxes": {
    "_comment": "Not provided.",
    "supported": false,
    "column": null
  },
  "reports": {
    "_comment": "Not exposed in this toy integration.",
    "supported": false,
    "format": null,
    "separate_csv": {
      "path": null,
      "filename_variants": [],
      "join_columns": []
    },
    "base_dir": null,
    "path_column": null
  },
  "extra_init_params": {},
  "hu_window": null,
  "custom_preprocessing": null
}"""

config_path = CONFIG_DIR / f"{REGISTRY_KEY}.json"
config_path.write_text(config_json)
print("wrote", config_path)
print(config_json)

wrote /home/fli40/RadHarmony/configs/datasets/my_montgomery.json
{
  "_comment": "Toy Montgomery County TB CXR set used in the RadHarmony datathon. A folder of PNG chest X-rays plus a small metadata CSV (data/montgomery_index.csv) with one row per image and columns pid / fname / tb. CSV-driven: the raw table already lists every image, so the harmonizer just maps columns. Full dataset source: https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Montgomery-County-CXR-Set/",
  "dataset_name": "My Montgomery (toy)",
  "registry_key": "my_montgomery",
  "modality": "2D",
  "image_format": "PNG",
  "base_image_dir": "<path_to_Montgomery>/MontgomerySet/CXR_png/",
  "primary_csv": {
    "_comment": "The metadata CSV that lists every image row.",
    "path": "data/montgomery_index.csv",
    "filename_variants": [
      "montgomery_index.csv"
    ]
  },
  "patient_id": {
    "_comment": "Direct column: pid. One study per patient in this set.",
    "column": "pid",
    "deriva

## Step 1: inspect the raw dataset

Look at what you have and decide how it maps onto the canonical columns
(`patient_id`, `study_id`, `image_path`, and the labels). This toy variant
ships a folder of PNGs plus a metadata CSV with one row per image; its
`pid` / `fname` / `tb` columns give the patient ID, the image filename, and
the TB label. That mapping is exactly what the config above records.


In [4]:
raw = pd.read_csv(INDEX_CSV)
print("rows:", len(raw), "| columns:", list(raw.columns))
display(raw.head())

rows: 138 | columns: ['pid', 'fname', 'tb']


,pid,fname,tb
0,1,MCUCXR_0001_0.png,0
1,2,MCUCXR_0002_0.png,0
2,3,MCUCXR_0003_0.png,0
3,4,MCUCXR_0004_0.png,0
4,5,MCUCXR_0005_0.png,0


## Step 2: the harmonizer

The harmonizer subclasses `BaseHarmonizer` and maps raw columns onto the
canonical ones. Because the raw CSV already lists every image, the default
`harmonize()` flow (read CSV, call the `_build_*` hooks) does the heavy lifting,
so we only override the three column builders. `tb` is already a column, so
listing it in `LABEL_COLS` is enough to carry it through.

The folder gets two files: the implementation and an `__init__.py` that
re-exports the class.


In [5]:
harmonizer_code = '''"""Harmonizer for a custom (toy) copy of the Montgomery County TB CXR set.

Datathon example of the *CSV-driven* harmonizer pattern: the raw dataset is a
folder of PNG chest X-rays plus a small metadata CSV
(``data/montgomery_index.csv``) with one row per image and columns
``pid`` / ``fname`` / ``tb``.

Raw CSV columns:
    pid    patient identifier (one study per patient here)
    fname  image filename, relative to base_image_dir
    tb     tuberculosis label (0 = normal, 1 = TB-positive)

Harmonized columns produced:
    patient_id, study_id, image_path, tb

The default :meth:`BaseHarmonizer.harmonize` flow already reads ``csv_path``
into ``self.df`` and calls the ``_build_*`` hooks, so this harmonizer only
overrides the three that map raw columns onto the canonical ones. ``tb`` is
already a column, so ``LABEL_COLS`` alone carries it into the output.
"""

import pandas as pd

from ..base import BaseHarmonizer


class MyMontgomeryHarmonizer(BaseHarmonizer):
    """Harmonize the toy Montgomery CSV into the standard RadHarmony format."""

    # Original CSV column names. The base class snake-cases them and keeps
    # them in the harmonized slice. Must be sorted alphabetically.
    LABEL_COLS = ["tb"]

    def _build_patient_id(self) -> None:
        # pid is the patient identifier; cast to str for stable grouping.
        self.df["patient_id"] = self.df["pid"].astype(str)

    def _build_study_id(self) -> None:
        # One study per patient in this set, so study_id mirrors patient_id.
        self.df["study_id"] = self.df["pid"].astype(str)

    def _build_image_path(self) -> None:
        # fname is already a bare filename relative to base_image_dir.
        self.df["image_path"] = self.df["fname"].astype(str)
'''

(HARMONIZER_DIR / "my_montgomery.py").write_text(harmonizer_code)
(HARMONIZER_DIR / "__init__.py").write_text(
    'from .my_montgomery import MyMontgomeryHarmonizer\n\n__all__ = ["MyMontgomeryHarmonizer"]\n'
)
print("wrote", HARMONIZER_DIR / "my_montgomery.py")
print("wrote", HARMONIZER_DIR / "__init__.py")
print(harmonizer_code)

wrote /home/fli40/RadHarmony/radharmony/harmonizer/my_montgomery/my_montgomery.py
wrote /home/fli40/RadHarmony/radharmony/harmonizer/my_montgomery/__init__.py
"""Harmonizer for a custom (toy) copy of the Montgomery County TB CXR set.

Datathon example of the *CSV-driven* harmonizer pattern: the raw dataset is a
folder of PNG chest X-rays plus a small metadata CSV
(``data/montgomery_index.csv``) with one row per image and columns
``pid`` / ``fname`` / ``tb``.

Raw CSV columns:
    pid    patient identifier (one study per patient here)
    fname  image filename, relative to base_image_dir
    tb     tuberculosis label (0 = normal, 1 = TB-positive)

Harmonized columns produced:
    patient_id, study_id, image_path, tb

The default :meth:`BaseHarmonizer.harmonize` flow already reads ``csv_path``
into ``self.df`` and calls the ``_build_*`` hooks, so this harmonizer only
overrides the three that map raw columns onto the canonical ones. ``tb`` is
already a column, so ``LABEL_COLS`` alone carr

## Step 3: the dataset class

The dataset subclasses `BaseRadiologicalDataset` and does three things:
declares `LABEL_COLS`, `SUPPORTED_OUTPUTS`, and `_HARMONIZER_CLS`; builds the
default 2-D transform; and implements `_get_harmonized_df()` to call the
harmonizer. The `@register_dataset("my_montgomery")` decorator makes it
resolvable by name. Everything else (patient-level splitting, k-fold, caching,
DataLoader support) is inherited.


In [1]:
dataset_code = '''"""MONAI dataset for a custom (toy) copy of the Montgomery County CXR set.

Datathon example that pairs with the CSV-driven :class:`MyMontgomeryHarmonizer`.
``base_image_dir`` points at the folder holding every PNG (the last common
folder); ``csv_path`` points at the metadata CSV. Only the classification
output (the binary TB label) is provided.
"""

import pandas as pd
import torch

from radharmony.harmonizer.my_montgomery import MyMontgomeryHarmonizer
from radharmony.registry import register_dataset
from radharmony.utils.infer import infer_path
from ..base import BaseRadiologicalDataset
from ..transforms import RadiologyTransform2D


@register_dataset("my_montgomery")
class MyMontgomeryDataset(BaseRadiologicalDataset):
    """PyTorch/MONAI dataset for the toy Montgomery CXR set.

    Args:
        base_image_dir: Folder containing every PNG (the last common folder),
            e.g. ``.../Montgomery-CXR/MontgomerySet/CXR_png/``. ``image_path``
            entries are bare filenames resolved relative to it.
        csv_path: Metadata CSV with pid / fname / tb columns. Auto-discovered
            near ``base_image_dir`` when ``None``.
        transform: MONAI Compose transform. Defaults to the 2-D 224 px pipeline.
        cache_dir: PersistentDataset cache directory. ``None`` disables caching.
        output_cls: Yield the TB label vector under the ``"cls"`` key.
        dtype: Output tensor dtype. Default ``torch.bfloat16``.
    """

    SUPPORTED_OUTPUTS: frozenset = frozenset({"cls"})
    LABEL_COLS = ["tb"]
    _HARMONIZER_CLS = MyMontgomeryHarmonizer

    def __init__(
        self,
        base_image_dir: str = None,
        csv_path: str = None,
        transform=None,
        cache_dir: str = "./cache",
        output_cls: bool = False,
        harmonized_df=None,
        harmonizer=None,
        harmonizer_path: str = None,
        dtype=torch.bfloat16,
    ):
        if (
            base_image_dir is None
            and harmonizer_path is None
            and harmonized_df is None
            and harmonizer is None
        ):
            raise ValueError(
                "base_image_dir is required when harmonizer_path is not provided."
            )

        # Auto-discover the metadata CSV near base_image_dir when not given.
        if harmonizer_path is None and harmonized_df is None and harmonizer is None:
            self._csv_path = (
                infer_path(base_image_dir, "montgomery_index.csv", user_path=csv_path or "")
                or csv_path
            )
        else:
            self._csv_path = csv_path

        if transform is None:
            output_keys = {"img"}
            if output_cls:
                output_keys.add("cls")
            transform = RadiologyTransform2D(
                img_size=224, output_keys=output_keys, dtype=dtype,
            ).get_transform()

        super().__init__(
            base_image_dir=base_image_dir,
            transform=transform,
            cache_dir=cache_dir,
            output_cls=output_cls,
            harmonized_df=harmonized_df,
            harmonizer=harmonizer,
            harmonizer_path=harmonizer_path,
        )

    def _get_harmonized_df(self) -> pd.DataFrame:
        preset = self._try_resolve_preset_harmonized()
        if preset is not None:
            return preset
        harmonizer = MyMontgomeryHarmonizer(csv_path=self._csv_path)
        return harmonizer.harmonize()
'''

(DATASET_DIR / "my_montgomery.py").write_text(dataset_code)
(DATASET_DIR / "__init__.py").write_text(
    'from .my_montgomery import MyMontgomeryDataset\n\n__all__ = ["MyMontgomeryDataset"]\n'
)
print("wrote", DATASET_DIR / "my_montgomery.py")
print("wrote", DATASET_DIR / "__init__.py")
print(dataset_code)

NameError: name 'DATASET_DIR' is not defined

## Step 4: wire up the exports

Two one-line edits let the classes be imported from the top-level packages, so
`from radharmony.dataset import MyMontgomeryDataset` works in any fresh session.
We append them idempotently.

These top-level edits take effect on the **next** kernel start (the package
objects are already imported in this session). To use the dataset right now,
Step 5 imports the new submodule directly, which runs the `@register_dataset`
decorator immediately.


In [ ]:
def append_line(path: Path, line: str):
    """Append `line` to a file if it is not already present."""
    existing = path.read_text() if path.exists() else ""
    if line.strip() in existing:
        print("already present in", path.name)
        return
    with path.open("a") as fh:
        if existing and not existing.endswith("\n"):
            fh.write("\n")
        fh.write(line if line.endswith("\n") else line + "\n")
    print("appended to", path)


append_line(
    PKG_DIR / "harmonizer" / "__init__.py",
    "from .my_montgomery import MyMontgomeryHarmonizer",
)
append_line(
    PKG_DIR / "dataset" / "__init__.py",
    "from .my_montgomery import MyMontgomeryDataset",
)

### Optional: the Gradio app

To browse the dataset in the RadHarmony app, add a build function and a registry
entry to `app.py`. This is not needed to use the dataset programmatically, so we
show the snippet rather than editing `app.py` here:

```python
# in app.py
def _build_my_montgomery(base_dir, csv_path, extra_field, extra_field2, cache_dir, **flags):
    csv_path = _infer(base_dir, "montgomery_index.csv", user_path=csv_path)
    if not csv_path:
        return None, "montgomery_index.csv is required."
    return (
        MyMontgomeryDataset(
            base_image_dir=base_dir,
            csv_path=csv_path,
            cache_dir=cache_dir or None,
            output_cls=flags.get("output_cls", False),
        ),
        None,
    )

# add to DATASET_REGISTRY
"My Montgomery (toy)": DatasetConfig(
    is_3d=False,
    extra_field_label="",
    build=_build_my_montgomery,
),
```


## Step 5: import and use it

Importing the new submodule runs the `@register_dataset` decorator, so the
dataset is live in this session. From there it behaves like any built-in:
resolve it by name, build it, and visualize a few samples.


In [ ]:
import importlib

import matplotlib.pyplot as plt

from radharmony.registry import resolve_dataset, list_datasets

# Import the freshly written submodule -> runs @register_dataset("my_montgomery").
importlib.import_module(f"radharmony.dataset.{REGISTRY_KEY}.{REGISTRY_KEY}")
print("registered:", REGISTRY_KEY in list_datasets())

MyMontgomeryDataset = resolve_dataset(REGISTRY_KEY)
my_ds = MyMontgomeryDataset(
    base_image_dir=MONTGOMERY_DIR,
    csv_path=INDEX_CSV,
    output_cls=True,
    cache_dir=None,
)
print("harmonized rows:", len(my_ds.get_harmonized_df()))

# visualize_samples yields one figure per sample; build, then display each.
built = my_ds.get_datasets()
for fig in my_ds.visualize_samples(built, n=3):
    display(fig)
    plt.close(fig)

## Step 6: the integrity check

The final step runs the same check CI uses: harmonize, load a subsample through
a `DataLoader`, and validate every sample's image shape, dtype, and value range,
plus the label vector. A clean `OK` report means the dataset is wired correctly.


In [ ]:
from radharmony.integrity import check_dataset, print_report, subsample_dataset

check_ds = MyMontgomeryDataset(
    base_image_dir=MONTGOMERY_DIR,
    csv_path=INDEX_CSV,
    output_cls=True,
    cache_dir=None,
)
subsample_dataset(check_ds, max_samples=30)  # fast pass over a subset
report = check_dataset(check_ds, num_workers=0)  # 0 workers is notebook-safe
print_report(report)

For a real contribution you would also register the dataset in
`scripts/check_datasets.py` so CI checks it on every change:

```python
# in get_specs()
S("my_montgomery", MyMontgomeryDataset, dict(
    base_image_dir=MY_MONTGOMERY_BASE, csv_path=MY_MONTGOMERY_CSV,
    output_cls=True)),
```


## What you built

You added `my_montgomery` to RadHarmony the production way. On disk:

- `configs/datasets/my_montgomery.json` (the config)
- `radharmony/harmonizer/my_montgomery/` (harmonizer + `__init__.py`)
- `radharmony/dataset/my_montgomery/` (dataset + `__init__.py`)
- appended exports in the two package `__init__.py` files

Unlike the inline version in notebook 2, this dataset survives a kernel restart,
loads by name anywhere, and is one wiki page away from a pull request. To finish
a real contribution, add a page under `docs/wiki/datasets/` and an inventory row
in `docs/wiki/datasets.md`, following the checklist in
[docs/add_dataset_skill.md](../docs/add_dataset_skill.md).
